In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pprint import pprint
from langchain import chat_models
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch

In [3]:
%pip install langchain_openai langchain langchain_core

  Using cached uuid_utils-0.14.0-cp39-abi3-win_amd64.whl.metadata (5.0 kB)
  Using cached langgraph-1.0.8-py3-none-any.whl.metadata (7.4 kB)
  Using cached langgraph_checkpoint-4.0.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_prebuilt-1.0.7-py3-none-any.whl.metadata (5.2 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 14.8 MB/s  0:00:00
   ---------------------------------------- 0.0/879.1 kB ? eta -:--:--
   ---------------------------------------- 879.1/879.1 kB 19.1 MB/s  0:00:00
Using cached uuid_utils-0.14.0-cp39-abi3-win_amd64.whl (182 kB)
Using cached langgraph-1.0.8-py3-none-any.whl (158 kB)
Using cached langgraph_checkpoint-4.0.0-py3-none-any.whl (46 kB)
Using cached langgraph_prebuilt-1.0.7-py3-none-any.whl (35 kB)

   ---------------- -----------------------  6/15 [openai]
   ---------------- -----------------------  6/15 [openai]
   ---------------- -----------------------  6/15

In [2]:
load_dotenv("C:/Users/Jitto/Documents/Code/AICoding/AiGenAiLearnJourney/.env")

True

In [3]:
llm = ChatOpenAI(
    model = 'gpt-4o',
    temperature = 0
)

In [6]:
def booking_handler(request: str) -> str:
    """Booking handler"""
    print("\n Booking hanlder")
    return f"Booking handler {request}"

In [7]:
def info_handler(request: str) -> str:
    """Info handler"""
    print("\n Info hanlder")
    return f"Info handler {request}"

def unclear_handler(request: str) -> str:
    """unclear handler"""
    print("\n unclear hanlder")
    return f"unclear handler {request}"    

In [16]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the users input and determine which handler shhould be processed.
        - If the request is related to booking flights or hotels, output 'booker'
        - If the request is related to gain info on any question or general info question, this is strictly restricted to general information about booking, output 'info'
        - If the request is unclear or doesnt fit the above two categories, output 'unclear' 
        
        Only output one word: 'booker', 'info', 'unclear'
        """),
    ("user", "{request}")
])

In [18]:
router_chain = router_prompt | llm | StrOutputParser()

In [19]:
router_chain.invoke({"request" : "Tell me about Quantum Physics"})

'unclear'

In [32]:
branches = {
    "booker" : RunnablePassthrough.assign(output = lambda x: booking_handler(x['request']['request'])),
    "info" : RunnablePassthrough.assign(output = lambda x: info_handler(x['request']['request'])),
    "unclear" : RunnablePassthrough.assign(output = lambda x: unclear_handler(x['request']['request'])),
}

In [33]:
delegation_branch = RunnableBranch(
    (lambda x: x['decision'].strip() == 'booker', branches["booker"]),
    (lambda x: x['decision'].strip() == 'info', branches['info']),
    branches['unclear']
)

In [34]:
complete_agent = {
    "decision" : router_chain,
    "request" : RunnablePassthrough()
} | delegation_branch | (lambda x: x['output'])

In [35]:
complete_agent.invoke({"request" : "Tell me about Quantum Physics"})


 unclear hanlder


'unclear handler Tell me about Quantum Physics'

### Better way to routing - Using agents

In [4]:
%pip install google.adk

  Using cached mcp-1.26.0-py3-none-any.whl.metadata (89 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_http-1.39.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached starlette-0.52.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached uvicorn-0.40.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached googleapis_common_protos-1.72.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl.metadata (2.4 kB)
  Using cached protobuf-6.33.5-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached python_multipart-0.0.22-py3-none-any.whl.metadata (1.8 kB)
  Using cached opentelemetry_exporter_otlp_proto_common-1.39.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached opentelemetry_proto-1.39.1-py3-none-any.whl.metadata (2.3 

In [6]:
import uuid
from typing import Dict, Any, Optional
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool
from google.genai import types
from google.adk.events import Event
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [8]:
booking_tool = FunctionTool(booking_handler)
info_tool = FunctionTool(info_handler)

In [11]:
booking_agent = Agent(
    model='gpt-4o',
    name = 'Booker',
    description = "A Agent that handles all the flight and hotel booking requests by calling the booking tool.",
    tools = [booking_tool]
)

In [8]:
@tool
def booking_handler(request: str) -> str:
    """Handles booking requests for flights and hotels."""
    print("---- Booking Handler Called ----")
    return f"Booking action for '{request}' has been simulated."

@tool
def info_handler(request: str) -> str:
    """Handles general information requests."""
    print("---- Info Handler Called ----")
    return f"Information request for '{request}'. Result: Simulated information retrieval."

In [13]:
llm_mini = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0
)

In [23]:
%pip install langchain_core.agents

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain_core.agents (from versions: none)
ERROR: No matching distribution found for langchain_core.agents


In [3]:
from langchain.agents import create_agent

In [14]:
tools = [booking_handler, info_handler]

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a coordinator. If the question is for flights or hotels, use booking handler.
                For all other information questions, use info_handler.
                Do not answer anything directly. Always use a tool"""),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name='agent_scrathpad')
])

In [17]:
agent = create_agent(model='gpt-4o-mini', tools=tools, system_prompt="""You are a coordinator. If the question is for flights or hotels, use booking handler.
                For all other information questions, use info_handler.
                Do not answer anything directly. Always use a tool""")

In [20]:
response = agent.invoke({"input" : "Book me a hotel in paris"})

In [24]:
response['messages'][-1]

AIMessage(content='How can I assist you today? Please let me know if you have questions about flights, hotels, or any other information!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 101, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_194c0b7559', 'id': 'chatcmpl-D9D2B5aaJq8SNvq5PrqzSkXbKVsrj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c5d09-7718-76a0-8b64-e564900119fa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 101, 'output_tokens': 26, 'total_tokens': 127, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
from langgraph.graph import StateGraph, END
from typing import TypedDict


In [26]:
class AgentState(TypedDict):
    input: str
    output: str

In [27]:

def router(state: AgentState):
    text = state["input"].lower()
    if "book" in text or "flight" in text or "hotel" in text:
        return "booker"
    return "info"

In [28]:
def booker_node(state: AgentState):
    return {"output": booking_handler(state["input"])}

def info_node(state: AgentState):
    return {"output": info_handler(state["input"])}

### Routing bw agents using LangChain

In [4]:
from langchain.chat_models import init_chat_model

In [7]:
model = init_chat_model('gpt-4o-mini')

In [6]:
from typing import Annotated, Literal, TypedDict
import operator

In [9]:
class AgentInput(TypedDict):
    """Simple input state for each subagent"""
    query: str

In [10]:
class AgentOutput(TypedDict):
    """Simple input state for each subagent"""
    query: str

In [11]:
class Classification(TypedDict):
    """A single routing deicision: which agent to call with query"""
    source: Literal['github', 'notion', 'slack']
    query: str

In [12]:
class RouterState(TypedDict):
    query: str
    classifications: list[Classification]
    results: Annotated[list[AgentOutput], operator.add]  # Reducer collects parallel results
    final_answer: str

In [13]:
from langchain.tools import tool


@tool
def search_code(query: str, repo: str = "main") -> str:
    """Search code in GitHub repositories."""
    return f"Found code matching '{query}' in {repo}: authentication middleware in src/auth.py"


@tool
def search_issues(query: str) -> str:
    """Search GitHub issues and pull requests."""
    return f"Found 3 issues matching '{query}': #142 (API auth docs), #89 (OAuth flow), #203 (token refresh)"


@tool
def search_prs(query: str) -> str:
    """Search pull requests for implementation details."""
    return f"PR #156 added JWT authentication, PR #178 updated OAuth scopes"


@tool
def search_notion(query: str) -> str:
    """Search Notion workspace for documentation."""
    return f"Found documentation: 'API Authentication Guide' - covers OAuth2 flow, API keys, and JWT tokens"


@tool
def get_page(page_id: str) -> str:
    """Get a specific Notion page by ID."""
    return f"Page content: Step-by-step authentication setup instructions"


@tool
def search_slack(query: str) -> str:
    """Search Slack messages and threads."""
    return f"Found discussion in #engineering: 'Use Bearer tokens for API auth, see docs for refresh flow'"


@tool
def get_thread(thread_id: str) -> str:
    """Get a specific Slack thread."""
    return f"Thread discusses best practices for API key rotation"

In [16]:
github_agent = create_agent(
    model,
    tools = [search_code, search_issues, search_prs],
    system_prompt = (
        "You are a GitHub expert. Answer questions about code, "
        "API references, and implementation details by searching "
        "repositories, issues, and pull requests."
    )
)

In [17]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4o")

github_agent = create_agent(
    model,
    tools=[search_code, search_issues, search_prs],
    system_prompt=(
        "You are a GitHub expert. Answer questions about code, "
        "API references, and implementation details by searching "
        "repositories, issues, and pull requests."
    ),
)

notion_agent = create_agent(
    model,
    tools=[search_notion, get_page],
    system_prompt=(
        "You are a Notion expert. Answer questions about internal "
        "processes, policies, and team documentation by searching "
        "the organization's Notion workspace."
    ),
)

slack_agent = create_agent(
    model,
    tools=[search_slack, get_thread],
    system_prompt=(
        "You are a Slack expert. Answer questions by searching "
        "relevant threads and discussions where team members have "
        "shared knowledge and solutions."
    ),
)

In [18]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

In [19]:
router_llm = init_chat_model(model='gpt-4o-mini')

In [20]:
# Define structured output schema for the classifier
class ClassificationResult(BaseModel):  
    """Result of classifying a user query into agent-specific sub-questions."""
    classifications: list[Classification] = Field(
        description="List of agents to invoke with their targeted sub-questions"
    )


In [21]:
(router_llm.with_structured_output(ClassificationResult)).invoke("Help check the git commit: 123")

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
C:\ProgramData\anaconda3\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ClassificationResult(clas... commit with ID 123.'}]), input_type=ClassificationResult])
  return self.__pydantic_serializer__.to_python(


ClassificationResult(classifications=[{'source': 'github', 'query': 'Check the git commit with ID 123.'}])

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart i

In [22]:
def classify_query(state: RouterState) -> dict:
    """Classify query and determine which agents to invoke."""
    structured_llm = router_llm.with_structured_output(ClassificationResult)  

    result = structured_llm.invoke([
        {
            "role": "system",
            "content": """Analyze this query and determine which knowledge bases to consult.
For each relevant source, generate a targeted sub-question optimized for that source.

Available sources:
- github: Code, API references, implementation details, issues, pull requests
- notion: Internal documentation, processes, policies, team wikis
- slack: Team discussions, informal knowledge sharing, recent conversations

Return ONLY the sources that are relevant to the query. Each source should have
a targeted sub-question optimized for that specific knowledge domain.

Example for "How do I authenticate API requests?":
- github: "What authentication code exists? Search for auth middleware, JWT handling"
- notion: "What authentication documentation exists? Look for API auth guides"
(slack omitted because it's not relevant for this technical question)"""
        },
        {"role": "user", "content": state["query"]}
    ])

    return {"classifications": result.classifications}

In [23]:
def route_to_agents(state: RouterState) -> dict:
    """Fan out to agent based on classifications"""
    return [
        Send(c['source'], {"query" : c['query']})
        for c in state['classifications']
    ]

In [35]:
def query_github(state: AgentInput) -> dict:
    """Query the GitHub Agent"""
    result = github_agent.invoke({
        "messages" : [{'role' : 'user', 'content' : state['query']}]
    })

    return {'results' : [{'source' : 'github', 'result' : result['messages'][-1].content}]}

In [31]:
def query_notion(state: AgentInput) -> dict:
    """Query the notion Agent"""
    result = notion_agent.invoke({
        "messages" : [{'role' : 'user', 'content' : state['query']}]
    })

    return {'results' : [{'source' : 'notion', 'result' : result['messages'][-1].content}]}

In [26]:
def query_slack(state: AgentInput) -> dict:
    """Query the slack Agent"""
    result = slack_agent.invoke({
        "messages" : [{'role' : 'user', 'content' : state['query']}]
    })

    return {'results' : [{'source' : 'slack', 'result' : result['messages'][-1].content}]}

In [36]:
def synthesize_results(state: RouterState) -> dict:
    """Combine results from all agents into a coherent answer"""
    if not state['results']:
        return {'final_answer' : 'No results found from any knowledge source.'}
    formatted = [
        f"**From {r['source'].title()}:**\n{r['result']}"
        for r in state['results']
    ]

    sysnthesis_response = router_llm.invoke([
        {
            'role' : 'system',
            'content' : f"""Synthesize these search results to answer the original question: {state['query']}"
            - Combine information from multiple sources without redundancy
            - Highlight the most relevant and actionable information
            - Note any discrepancies between sources
            - Keep the content concise and well-organized
            """
        },
        {
            'role' : 'user',
            'content' : '\n\n'.join(formatted)
        }
    ])

    return {'final_answer' : sysnthesis_response.content}

In [37]:
workflow = (
    StateGraph(RouterState)
    .add_node("classify", classify_query)
    .add_node("github", query_github)
    .add_node("notion", query_notion)
    .add_node("slack", query_slack)
    .add_node("synthesize", synthesize_results)
    .add_edge(START, "classify")
    .add_conditional_edges("classify", route_to_agents, ["github", "notion", "slack"])
    .add_edge("github", "synthesize")
    .add_edge("notion", "synthesize")
    .add_edge("slack", "synthesize")
    .add_edge("synthesize", END)
    .compile()
)

In [38]:
if __name__ == "__main__":
    result = workflow.invoke({
        "query": "How do I authenticate API requests?"
    })

C:\ProgramData\anaconda3\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ClassificationResult(clas... for API auth guides'}]), input_type=ClassificationResult])
  return self.__pydantic_serializer__.to_python(


In [39]:
result

{'query': 'How do I authenticate API requests?',
 'classifications': [{'source': 'github',
   'query': 'What authentication code exists? Search for auth middleware, JWT handling'},
  {'source': 'notion',
   'query': 'What authentication documentation exists? Look for API auth guides'}],
 'results': [{'source': 'github',
   'result': "Here's what I found regarding authentication code:\n\n1. **Auth Middleware**: There is an authentication middleware implemented in `src/auth.py`. This typically includes functions or classes that handle user authentication, session verification, and request validation.\n\n2. **JWT Handling**: The same file, `src/auth.py`, also contains code for handling JSON Web Tokens (JWT). This usually involves creating, decoding, and validating JWTs for secure API communications.\n\nFor more specific implementation details or examples, you might want to look at the particular functions or classes defined in `src/auth.py`."},
  {'source': 'notion',
   'result': 'I found

## Parellization

In [43]:
from langchain_core.runnables import Runnable, RunnableParallel, RunnablePassthrough
llm = ChatOpenAI(model='gpt-4o-mini')

In [44]:
summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ('system', 'Summarize the following topic: Consiely:'),
        ('user', '{topic}')
    ])
    | llm
    | StrOutputParser()
)

In [46]:
questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ('system', 'Generate three interesting questions about the following topic:'),
        ('user', '{topic}')
    ])
    | llm
    | StrOutputParser()
)

In [47]:
terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ('system', 'Identify 5-10 key terms from the following topic,separated by commas:'),
        ('user', '{topic}')
    ])
    | llm
    | StrOutputParser()    
)

In [48]:
map_chain = RunnableParallel(
    {
        "summary": summarize_chain, 
        "questions" :questions_chain,
        "key_terms" : terms_chain,
        "topic" : RunnablePassthrough()
    }
)

In [51]:
sys_prompt = ChatPromptTemplate.from_messages([
    ('system', """Based on the following information: 
        Summar: {summary}
        Related Questions: {questions}
        Key Terms: {key_terms}
        Synthesize a comprehensive answer. """),
    ('user', "original topic: {topic}")
])

In [52]:
full_parallel_chain = map_chain | sys_prompt | llm | StrOutputParser()

In [53]:
async def run_parallel(topic: str):
    response = await full_parallel_chain.ainvoke(topic)
    print(response)

In [57]:
import asyncio

In [62]:
if __name__ == "__main__":
    test_topic = "The history of space exploration"
    # In Python 3.7+, asyncio.run is the standard way to run an asyncfunction.
    asyncio.run(run_parallel(test_topic))

RuntimeError: asyncio.run() cannot be called from a running event loop

In [56]:
%pip install asyncio

Note: you may need to restart the kernel to use updated packages.
